In [1]:
from pathlib import Path
import pandas as pd
import nrrd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
root_dir = Path('/home/lang/Data/TCIA/head_neck/HN1/medical_data/')
data_dir = root_dir / 'nrrd'

In [3]:
path_df = pd.DataFrame(columns=['Patient', 'CT_path', 'GTV_path'])
for ii, cc in enumerate([dir for dir in data_dir.iterdir()]):
    gtv_path = [str(ff) for ff in (cc / 'RTSTRUCT').glob('GTV-1*')]
    if len(gtv_path) != 1:
        raise ValueError('found non/several gtv files for {}'.format(cc))
    ct_path = [str(ff) for ff in (cc / 'CT').glob('*nrrd')]
    if len(ct_path) != 1:
        raise ValueError('found non/several gtv files for {}'.format(cc))
        
    path_df.loc[ii] = [cc.name, *ct_path, *gtv_path]

In [4]:
path_df.head()

,Patient,CT_path,GTV_path
0,HN1118,/home/lang/Data/TCIA/head_neck/HN1/medical_dat...,/home/lang/Data/TCIA/head_neck/HN1/medical_dat...
1,HN1046,/home/lang/Data/TCIA/head_neck/HN1/medical_dat...,/home/lang/Data/TCIA/head_neck/HN1/medical_dat...
2,HN1703,/home/lang/Data/TCIA/head_neck/HN1/medical_dat...,/home/lang/Data/TCIA/head_neck/HN1/medical_dat...
3,HN1308,/home/lang/Data/TCIA/head_neck/HN1/medical_dat...,/home/lang/Data/TCIA/head_neck/HN1/medical_dat...
4,HN1029,/home/lang/Data/TCIA/head_neck/HN1/medical_dat...,/home/lang/Data/TCIA/head_neck/HN1/medical_dat...


In [5]:
save = False
if save:
    path_df.to_csv(root_dir / 'HN1_all_files.csv', index=False)

In [6]:
size_df = pd.DataFrame(columns=['patient_ID', 'trans_space', 'long_space', 'voxel_number'])

for idx, row in path_df.reset_index(drop=True).iterrows():

    sgmt, header = nrrd.read(row['GTV_path'])
   
    space = header['space directions']
    space = space[np.where(space)]
    if len(space) != 3:
        raise ValueError('spaceion unequal 3')
        
    if abs(space[0] - space[1]) > 0.01:
        raise ValueError(f'transversal spaceions differ {space[0]} vs {space[1]}')
    else:
        trans_space = space[0]
        
    gtv_voxel = np.count_nonzero(sgmt)
    
    size_df.loc[idx] = [row['Patient'], trans_space, space[-1], gtv_voxel]

In [7]:
size_df['volume'] = size_df['trans_space']**2 * size_df['long_space'] * size_df['voxel_number']

In [8]:
size_df.head()

,patient_ID,trans_space,long_space,voxel_number,volume
0,HN1118,0.976562,3.0,2997,8574.49
1,HN1046,0.977000,3.0,4940,14146.1
2,HN1703,0.977000,3.0,13875,39732.3
3,HN1308,0.977000,3.0,3509,10048.3
4,HN1029,0.976562,3.0,2490,7123.95


In [9]:
for direct in ['trans_space', 'long_space', 'volume']:
    q75, q25 = np.percentile(size_df[direct], [75 ,25])
    mean = size_df[direct].mean()
    print(f'{direct}\t mean: {mean:.2f}, q25: {q25:.2f}, q75: {q75:.2f}')

trans_space	 mean: 0.98, q25: 0.98, q75: 0.98
long_space	 mean: 2.99, q25: 3.00, q75: 3.00
volume	 mean: 27200.60, q25: 3690.72, q75: 33708.57


In [10]:
save = False
if save:
    size_df.to_csv(root_dir / 'HN1_sizes.csv', index=False)